Yes. For your peer discussion, I would **not focus on the regression details or every plot**. The most useful thing to explain is:

> **Which parts of this notebook are reusable/automated, which parts require human judgment, and what must be reconsidered when applying the pipeline to a different dataset.**

Your notebook already contains a substantial automated pipeline, but several decisions are inherently corpus-specific. 

### What is automated and reusable

Once the inputs and configuration are correct, the code can automatically handle data preprocessing, DTM construction, vocabulary filtering, train/test splitting, LDA fitting, extraction of \(\beta\) and \(\theta\), calculation of perplexity/coherence/topic similarity, representative-document extraction, probability-weighted topic prevalence, annual prevalence, and plotting.

So you can tell your peers:

> “The computational pipeline is largely reusable. If I provide another pair of corpora in the expected format, most preprocessing, LDA estimation, diagnostics, prevalence calculations, temporal aggregation, and visualization can run automatically.”

The current pipeline also deliberately applies identical preprocessing to Overton and Scopus and uses a proportional DF threshold rather than an identical absolute threshold because the corpora have different sizes. 

### What cannot simply be automated

There are four major human-judgment stages.

**First is domain relevance screening.** Your power-system dictionary was designed for this particular research question. It cannot automatically be reused for, say, healthcare AI or transportation optimization. The vocabulary and screening criteria would have to be redesigned and manually validated. In your study, manual inspection of retained and rejected documents was important before freezing the screening rule. 

**Second is topic-number selection.** The code can calculate perplexity, coherence and similarity for different \(K\), but it cannot determine the substantively “correct” number of topics by itself. Your own experiment demonstrated why: perplexity kept improving toward \(K=100\), while coherence deteriorated and topics became increasingly fragmented. You therefore inspected topic interpretability and selected \(K=50\) as a useful thematic resolution. 

**Third is topic labeling.** This is probably the biggest manual stage. LDA gives:

$$
\text{Topic 1} =
\{\text{word probabilities}\}
$$

It does not tell you:

> “This is Power-System Flexibility and Grid Planning.”

We assigned labels by examining both the top terms and the five most representative publications. A human needs to judge whether those documents actually form a coherent concept. Some topics were very clear; others were heterogeneous and needed deliberately broad labels.

**Fourth is meta-theme mapping.** The mapping from 50 Overton + 50 Scopus latent topics into 11 common meta-themes is an interpretive coding exercise. The code can aggregate the probabilities once the mapping exists, but it cannot scientifically decide by itself that a topic belongs to *Optimization and Computational Methods* rather than *Machine Learning and Data-Driven Methods*.

### If somebody uses your notebook with different data

This is the key warning I'd give them:

> **Do not just change the filenames and press “Run All.”**

They should reconsider at least these things:

| Component                          | Reusable automatically? | Needs reconsideration with new data?   |
| ---------------------------------- | ----------------------- | -------------------------------------- |
| Deduplication code                 | Mostly                  | **Yes — identifiers/data structure**   |
| Domain screening                   | No                      | **Definitely**                         |
| Stopwords                          | Partly                  | **Yes — especially query words**       |
| Stemming/preprocessing             | Mostly                  | Check language/domain                  |
| Minimum DF                         | Formula reusable        | **Recalculate from corpus size**       |
| Train/test split                   | Yes                     | Usually no                             |
| Candidate \(K\) search             | Yes                     | **Run again**                          |
| Final \(K=50\)                     | **No**                  | **Select again**                       |
| LDA Gibbs parameters               | Mostly                  | Check convergence/computation          |
| Topic prevalence                   | Yes                     | No                                     |
| Representative-document extraction | Yes                     | No                                     |
| Topic labels                       | **No**                  | **Manual interpretation**              |
| Meta-theme definitions             | Partly                  | **Manual/research-question dependent** |
| Topic → meta-theme mapping         | **No**                  | **Manual validation**                  |
| Temporal window                    | **No**                  | **Depends on annual coverage**         |
| Figures                            | Yes                     | May need formatting/range changes      |

### In particular, these values are NOT universal constants

Make this very clear to your peers:

```text
MIN_DOC_PERCENT = 0.002
K = 50
2004–2025
11 meta-themes
power-system domain dictionary
query-specific stopwords
```

They are methodological decisions for **this analysis**, not settings that should automatically be used on every dataset.

For example, your \(0.2\%\) DF rule produced minimum DF values of 11 for Overton and 25 for Scopus because their corpus sizes differ. 

Similarly, \(K=50\) should not be hard-coded as “the correct number of topics.” Your notebook explicitly arrived at it after comparing multiple candidate resolutions. 

### What should always be manually inspected

I would tell them that at minimum they should manually inspect:

1. **Random retained/rejected documents after domain screening** — does the filter actually represent the intended domain?
2. **Top terms for candidate LDA models** — are topics coherent or fragmented?
3. **Representative documents for every final topic** — do the documents support the proposed label?
4. **Ambiguous/noisy topics** — don't force them into artificially specific labels.
5. **Topic-to-meta-theme assignments** — especially topics that plausibly belong to multiple categories.
6. **Temporal coverage** — don't let years with only a handful of publications drive trend conclusions.
7. **Final figures and conclusions** — check whether the numerical result actually supports the narrative.

That is where human domain expertise enters the workflow.

### The simplest way to explain the philosophy

You could tell your peers:

> “I separate the workflow into computational steps and interpretive steps. The computer handles reproducible operations such as preprocessing, DTM construction, LDA estimation, probability calculations, ranking representative documents, prevalence aggregation, and visualization. Human judgment is required for defining relevance, validating the corpus, choosing a useful topic resolution, interpreting and labeling topics, creating common meta-themes, and deciding whether the final results make substantive sense. If the dataset or research question changes, those judgment-based stages must be revisited rather than simply reusing my current settings.”

And there is an especially important principle behind your notebook:

$$
\boxed{\text{Automate calculation; validate interpretation manually.}}
$$

That's probably the most useful message to give your peers about how this notebook should be reused.


For your peers, I would explain it as a **methodological pipeline and the reasoning behind each decision**, rather than walking through every notebook cell.

You can present it roughly like this:

### 1. Research objective

> “The goal is to compare the research themes emphasized in policy-facing literature versus academic literature for power systems, machine learning, and optimization. Overton represents the policy-facing side, while Scopus represents the academic side. I model the two corpora independently and later compare their topic structures.”

Your original search concept was:

> `"Power flow" AND "Machine learning" AND (Optimization OR Optimisation)`

### 2. Initial data cleaning

Explain that the raw database exports contained duplicates, particularly because the same publication could occur multiple times.

For Overton, you found that DOI alone was not always a safe publication identifier because some DOI values corresponded to multiple titles, especially books/book chapters. Therefore, you deduplicated using the publication identity more carefully rather than blindly collapsing every repeated DOI.

For Scopus, you also handled invalid DOI values such as `"0"` and genuine duplicates.

The resulting datasets were then used as the starting publication corpora.

### 3. Important problem discovered: the corpora were not comparable

This is one of the most important parts to explain.

You initially ran LDA on the retrieved corpora and found topics involving unrelated areas such as biology, materials, family/social topics, etc., particularly in Overton.

You then performed a domain-relevance diagnostic and found that the original Overton corpus was substantially more heterogeneous than Scopus.

So you changed the design to answer the more precise question:

> **“What topics characterize power-system/ML/optimization research appearing in Overton versus Scopus?”**

That meant both datasets needed to be restricted to approximately the **same substantive power/energy domain** before comparing their topics.

### 4. Common domain-relevance screening

You developed a common lexical relevance screen and applied **the same rule to both corpora**.

The screen covered concept families such as:

* power systems and grids;
* power flow and system operation;
* transmission/distribution;
* electricity;
* generation and DERs;
* renewables;
* storage and EVs;
* power electronics;
* load/demand;
* energy systems;
* electricity markets, reliability and related power-system terminology.

Importantly, you did **not** use broad standalone words such as `system`, `network`, `energy`, or `optimization`, because they would admit too much unrelated research.

You validated the screen manually by sampling both retained and rejected publications. The retained samples were predominantly power/energy research, while rejected samples were largely unrelated.

Then you froze the screening rule rather than continuing to tune it to individual papers.

Your final domain-filtered corpora were approximately:

> **Overton: 5,045 documents**
> **Scopus: 12,042 documents**

You should emphasize:

> “I did not force the datasets to have the same size. I applied the same relevance criterion to both.”

### 5. Text preprocessing

You used **R `tm` and `SnowballC`**, consistently for both corpora.

The preprocessing pipeline included:

> lowercase → punctuation removal → number removal → whitespace normalization → English stopword removal → query-specific stopword removal → English stemming.

You deliberately removed the original query terms:

```text
power
flow
machine
learning
optimization
optimisation
```

because otherwise LDA could simply rediscover the search query rather than identifying more informative themes.

After preprocessing:

> Overton: 5,045 documents, ~565k tokens before DTM filtering
> Scopus: 12,042 documents, ~1.51M tokens before DTM filtering.

### 6. DTM and vocabulary filtering

This is another methodological point worth explaining carefully.

Because the corpora have different sizes, you decided **not to use the same absolute minimum document frequency**.

For example, DF ≥ 25 would mean roughly 0.5% of Overton but only 0.2% of Scopus.

Instead, you used approximately the same **relative document-frequency threshold: 0.2% of documents**.

That produced:

|                     | Overton |    Scopus |
| ------------------- | ------: | --------: |
| Documents           |   5,045 |    12,042 |
| Minimum DF          |      11 |        25 |
| Effective threshold |  0.218% |    0.208% |
| Vocabulary          |   2,865 |     2,737 |
| Tokens              | 512,540 | 1,384,122 |

This is a strong point to mention because the resulting vocabularies are comparable without artificially forcing them to have equal sizes.

### 7. LDA model selection

You used **R `topicmodels` with Gibbs sampling**.

For each corpus, you made a reproducible:

> 80% training / 20% held-out split, seed = 123.

That gave:

> Overton: 4,036 training / 1,009 held-out
> Scopus: 9,633 training / 2,409 held-out.

Then you performed a cheap screening over:

$$
K=5,10,15,20,25,30,40,50,60,70,80,100.
$$

using short Gibbs chains:

```text
burn-in = 50
iterations = 100
thin = 10
```

Held-out perplexity kept decreasing through \(K=100\) for **both corpora**.

This is important:

> “I did not simply choose K=100 because it had the lowest perplexity. Perplexity continued rewarding increasing model complexity, so I treated it as only one model-selection criterion.”

### 8. Multi-criterion model selection

You therefore evaluated:

$$
K=\{30,50,70,100\}
$$

with substantially longer candidate chains:

```text
burn-in = 500
iterations = 1000
thin = 50
```

and compared:

* held-out perplexity;
* semantic coherence;
* inter-topic cosine similarity;
* topic interpretability/granularity.

The pattern was consistent in both corpora:

> Increasing K improved perplexity and reduced average topic similarity, but semantic coherence deteriorated and topics became increasingly fragmented.

For example, Scopus moved from mean coherence around −63 at K=30 to −78 at K=100, while perplexity improved from ~560 to ~454.

Then you manually inspected top terms. At K=30, some important concepts remained aggregated; at K=70/100, themes became increasingly narrow or fragmented. K=50 gave a useful intermediate resolution.

So your current selection is:

$$
\boxed{K_{\text{Overton}}=50,\qquad K_{\text{Scopus}}=50}
$$

A useful sentence for your peers is:

> “The fact that both ended at 50 was not imposed in advance. I applied the same model-selection procedure independently to each corpus, and 50 was retained as the working interpretive resolution for both.”

Also avoid calling 50 a mathematically proven “optimal K.” Say **selected/preferred thematic resolution**.

### 9. Final LDA estimation

Once K was fixed, you stopped using the held-out split and refitted the final model on **all documents** in each corpus.

The final configuration is:

```text
K          = 50
seed       = 123
burn-in    = 2,000
iterations = 10,000
thin       = 100
```

Your final Overton model has already completed successfully:

> 5,045 documents
> 2,865 terms
> 512,540 tokens
> K = 50
> runtime ≈ 397 seconds.

The final model exports:

```text
beta.csv
theta.csv
top_terms.csv
document_topics.csv
topic_prevalence.csv
model_summary.csv
```

Explain `beta` and `theta` simply:

> **β (beta)** tells us which words characterize each topic.
> **θ (theta)** tells us how strongly each topic is represented in each publication.

### 10. What comes next

This is the part your peers will probably care about most:

> “The statistical modeling is almost finished. The next stage is substantive interpretation and comparison.”

Once the final Scopus model is finished, you will:

1. label the 50 Overton and 50 Scopus topics from their highest-probability terms and representative documents;
2. calculate topic prevalence;
3. identify corresponding themes between the two corpora;
4. compare their relative prevalence.

Then you can make statements of the form:

> **Policy-facing literature (Overton)** places relatively greater emphasis on themes A, B, and C, whereas **academic literature (Scopus)** places relatively greater emphasis on themes D, E, and F.

The key is that those conclusions will come **after** topic matching and prevalence comparison, not merely from looking at the top words.

### A short version you can actually say in the meeting

> “I collected two corpora, Overton for policy-facing literature and Scopus for academic literature, with the goal of comparing their research themes in power systems, machine learning, and optimization. An initial LDA run showed that Overton contained a lot of off-domain material, so before doing the final topic modeling I developed and manually validated a common power-and-energy relevance filter and applied exactly the same criterion to both datasets. That left about 5,000 Overton and 12,000 Scopus publications.
>
> “I preprocess both datasets identically in R using `tm` and stemming, and I remove the original search-query terms so the topics aren't dominated by the words used to retrieve the documents. Because the corpora have different sizes, I use an approximately 0.2% minimum document-frequency threshold rather than the same absolute count, resulting in about 2,800 terms in each vocabulary.
>
> “For LDA model selection, I use an 80/20 train-held-out split and screen K from 5 to 100. Perplexity keeps decreasing as K increases, so I don't use perplexity alone. I then compare K=30, 50, 70 and 100 using longer Gibbs chains, semantic coherence, topic similarity, and interpretability. Higher K improves predictive fit but reduces coherence and increasingly fragments the themes. For now I've selected K=50 independently for both corpora.
>
> “I'm now fitting the final K=50 models on the complete corpora using much longer Gibbs chains. Overton is finished, and Scopus is next. After that, I'll label the topics, calculate their prevalence, match comparable themes across Overton and Scopus, and quantify which themes are emphasized more in policy-facing literature versus academia.”

That is the story I would present. It explains not only **what you did**, but also **why the pipeline changed**, which is probably the most valuable part of the notebook for a peer discussion.
